# Deep Incremental Image Retrieval - IP102 on Kaggle

This notebook runs incremental image retrieval training on the IP102 dataset.
It auto-clones the repository from GitHub and runs the training pipeline.

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path

# Setup paths
KAGGLE_INPUT_DIR = '/kaggle/input'
KAGGLE_WORKING_DIR = '/kaggle/working'
REPO_DIR = os.path.join(KAGGLE_WORKING_DIR, 'Deep-Incremental-Image-Retrieval')

print(f"Kaggle input dir: {KAGGLE_INPUT_DIR}")
print(f"Working dir: {KAGGLE_WORKING_DIR}")
print(f"Repo dir: {REPO_DIR}")

In [ ]:
# Auto-clone repository from GitHub
REPO_URL = 'https://github.com/nta2112/Deep-Incremental-Image-Retrieval-for-IP102.git'
repo_url = os.environ.get('IP102_CODE_REPO', REPO_URL)

if not os.path.exists(REPO_DIR):
    print(f"Cloning repository from {repo_url}...")
    result = subprocess.run(['git', 'clone', repo_url, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Clone failed: {result.stderr}")
        raise RuntimeError("Repository not found. Set IP102_CODE_REPO env var.")
    else:
        print("Repository cloned successfully")
else:
    print("Repository already exists, pulling latest changes...")
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True)

In [ ]:
# Find dataset directory (single input)
def find_dataset_root():
    """Auto-find IP102 dataset root directory"""
    # Primary path from user
    primary_path = '/kaggle/input/datasets/nta212/ip102-for-object-detection'
    if os.path.exists(os.path.join(primary_path, 'train.json')):
        return primary_path
    
    # Fallback candidates
    candidates = [
        os.path.join(KAGGLE_INPUT_DIR, 'ip102-dataset'),
        os.path.join(KAGGLE_INPUT_DIR, 'ip102'),
        os.path.join(KAGGLE_INPUT_DIR, 'IP102 dataset'),
    ]
    
    for candidate in candidates:
        if os.path.exists(os.path.join(candidate, 'train.json')):
            return candidate
    
    # Deep walk search
    for root, dirs, files in os.walk(KAGGLE_INPUT_DIR):
        if 'train.json' in files and 'filtered_class.txt' in files:
            return root
    
    raise FileNotFoundError("IP102 dataset not found in Kaggle inputs")

DATA_ROOT = find_dataset_root()
print(f"Dataset root: {DATA_ROOT}")

# Verify dataset structure
for f in ['train.json', 'val.json', 'test.json', 'filtered_class.txt', 'classes.txt']:
    path = os.path.join(DATA_ROOT, f)
    print(f"  {f}: {'✓' if os.path.exists(path) else '✗'} {path}")

voc_images = os.path.join(DATA_ROOT, 'VOC2007', 'VOC2007', 'JPEGImages')
print(f"  VOC2007/JPEGImages: {'✓' if os.path.exists(voc_images) else '✗'} {voc_images}")

In [ ]:
# Add repo to Python path
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Pretrained weight path
PRETRAINED_WEIGHT_PATH = '/kaggle/input/models/nhannguyen5578/deep-incremental-image-retrieval-pretrain/pytorch/default/1/bn_inception-52deb4733.pth'
if os.path.exists(PRETRAINED_WEIGHT_PATH):
    print(f"Pretrained weight found: {PRETRAINED_WEIGHT_PATH}")
else:
    print(f"WARNING: Pretrained weight not found at {PRETRAINED_WEIGHT_PATH}")
    PRETRAINED_WEIGHT_PATH = None

# Verify imports work
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
def run_train(model_name='BN_Inception', max_tasks=4, memory_size=0, 
             epochs=100, batch_size=32, lr=1e-5, dim=512,
             gpu_ids='auto', save_dir=None, pretrained_path=None):
    """
    Run incremental training on IP102 dataset.
    
    Args:
        model_name: Model architecture ('BN_Inception')
        max_tasks: Number of incremental tasks (default 4 for IP102)
        memory_size: Memory size for replay (0 = no replay)
        epochs: Number of epochs per task
        batch_size: Batch size per GPU
        lr: Learning rate
        dim: Embedding dimension
        gpu_ids: GPU IDs to use ('auto' for all available)
        save_dir: Directory to save checkpoints and logs
        pretrained_path: Path to pretrained BN_Inception weight
    
    Returns:
        Path to results.csv
    """
    import argparse
    import torch
    from train import main
    
    if save_dir is None:
        save_dir = os.path.join(KAGGLE_WORKING_DIR, 'checkpoints', f'{model_name}_ip102_tasks{max_tasks}')
    
    if gpu_ids == 'auto':
        gpu_ids = ','.join(str(i) for i in range(torch.cuda.device_count()))
    
    if pretrained_path is None:
        pretrained_path = PRETRAINED_WEIGHT_PATH
    
    # Build args namespace
    args = argparse.Namespace(
        lr=lr,
        batch_size=batch_size,
        num_instances=4,
        dim=dim,
        width=224,
        origin_width=256,
        ratio=0.16,
        alpha=30,
        beta=0.1,
        orth_reg=1.0,
        k=16,
        margin=0.5,
        init='random',
        Incremental_flag=True,
        data='ip102',
        freeze_BN=True,
        data_root=DATA_ROOT,
        net=model_name,
        loss='HardMining',
        epochs=epochs,
        save_step=10,
        resume=pretrained_path,
        resume_pre_step_2=None,
        print_freq=10,
        save_dir=save_dir,
        nThreads=4,
        momentum=0.9,
        weight_decay=5e-4,
        loss_base=0.75,
        gpu_ids=gpu_ids,
        max_tasks=max_tasks,
        gallery_eq_query=True
    )
    
    print(f"Starting training with args:")
    for k, v in vars(args).items():
        print(f"  {k}: {v}")
    
    main(args)
    
    results_csv = os.path.join(save_dir, 'results.csv')
    history_json = os.path.join(save_dir, 'history.json')
    
    print(f"\nTraining completed!")
    print(f"Results: {results_csv}")
    print(f"History: {history_json}")
    
    return results_csv, history_json

In [ ]:
# Quick test run (1 epoch per task, 2 tasks)
print("="*60)
print("QUICK TEST RUN: 2 tasks, 1 epoch each")
print("="*60)

results_csv, history_json = run_train(
    model_name='BN_Inception',
    max_tasks=2,
    epochs=1,
    batch_size=16,
    lr=1e-5,
    gpu_ids='auto',
    pretrained_path=PRETRAINED_WEIGHT_PATH
)

In [ ]:
# Full training run (uncomment to run)
# print("="*60)
# print("FULL TRAINING RUN: 4 tasks, 100 epochs each")
# print("="*60)
# 
# results_csv, history_json = run_train(
#     model_name='BN_Inception',
#     max_tasks=4,
#     epochs=100,
#     batch_size=32,
#     lr=1e-5,
#     gpu_ids='auto',
#     pretrained_path=PRETRAINED_WEIGHT_PATH
# )

In [ ]:
# Display results.csv
import pandas as pd

results_path = os.path.join(KAGGLE_WORKING_DIR, 'checkpoints', 'BN_Inception_ip102_tasks2', 'results.csv')

if os.path.exists(results_path):
    df = pd.read_csv(results_path)
    print("Results:")
    print(df.to_string(index=False))
    
    # Also display history.json
    history_path = os.path.join(KAGGLE_WORKING_DIR, 'checkpoints', 'BN_Inception_ip102_tasks2', 'history.json')
    if os.path.exists(history_path):
        with open(history_path, 'r') as f:
            history = json.load(f)
        print("\nHistory:")
        print(json.dumps(history, indent=2))
else:
    print(f"Results file not found at {results_path}")
    print("Available files in checkpoints:")
    for root, dirs, files in os.walk(os.path.join(KAGGLE_WORKING_DIR, 'checkpoints')):
        for f in files:
            print(f"  {os.path.join(root, f)}")

In [ ]:
# Unit test metrics
from evaluations.metrics import test_metrics_perfect_case, test_metrics_all_seen

print("Running metric unit tests...")
test_metrics_perfect_case()
test_metrics_all_seen()
print("All metric tests passed!")